# 🔍 Phase 3 — EDA Complète (Roadmap 8 Étapes)
## Projet Capstone : Prédiction des Prix Immobiliers en Mauritanie
### Master 1 Machine Learning — SupNum

---

> **Objectif :** Appliquer la roadmap EDA (8 étapes vue en cours) sur le dataset immobilier de Nouakchott.  
> **Input :** `enriched_data.csv` (sortie Phase 2) ou `kaggle_train.csv`  
> **Output :** Dataset nettoyé, analysé et prêt pour la modélisation (Phase 4)

| Étape | Contenu |
|-------|---------|
| 1 | Chargement & Inspection initiale |
| 2 | Nettoyage de base |
| 3 | Valeurs manquantes (MCAR / MAR / MNAR) |
| 4 | Détection d'outliers |
| 5 | Analyse univariée |
| 6 | Analyse bivariée |
| 7 | Analyse multivariée |
| 8 | Préparation finale |

---

## 🔧 Setup — Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, kruskal, shapiro

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('✅ Bibliothèques importées')

---
## 📥 Étape 1 : Chargement & Inspection initiale

> **Objectif :** Comprendre la structure — dimensions, types, aperçu, doublons, cohérence des formats.

In [ ]:
# ── Chargement ────────────────────────────────────────────────────────────────
# Adapter selon votre structure :
# df = pd.read_csv('data/processed/enriched_data.csv')  # Si Phase 2 terminée
df = pd.read_csv('kaggle_train.csv')  # Dataset Kaggle

print(f'📐 Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print(f'📊 Mémoire : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
df.head()

In [ ]:
# ── Types de variables ─────────────────────────────────────────────────────────
print('📋 Types de variables :')
print(df.dtypes.value_counts())
print(f'\n🔢 Numériques : {df.select_dtypes(include="number").shape[1]}')
print(f'🏷️  Catégorielles/Texte : {df.select_dtypes(include="object").shape[1]}')

print('\n📋 Détail par colonne :')
for col in df.columns:
    print(f'  {col:25s} → {str(df[col].dtype):10s} | {df[col].nunique():5d} uniques | NaN: {df[col].isna().sum()}')

In [ ]:
df.info()

In [ ]:
# ── Valeurs uniques ────────────────────────────────────────────────────────────
print('🔑 Valeurs uniques par variable :')
for col in df.columns:
    print(f'  {col:25s} → {df[col].nunique():5d} uniques')
    if df[col].dtype == 'object' and df[col].nunique() <= 15:
        print(f'    Valeurs: {df[col].unique().tolist()}')

In [ ]:
# ── Doublons ───────────────────────────────────────────────────────────────────
cols_no_id = [c for c in df.columns if c != 'id']
duplicates = df[cols_no_id].duplicated()
print(f'🔁 Doublons (hors id) : {duplicates.sum()}')
if duplicates.sum() > 0:
    print('\nLignes dupliquées :')
    print(df[duplicates].head())

In [ ]:
# ── Cohérence des formats ──────────────────────────────────────────────────────
print('📅 Vérification des formats :')

# Prix : doit être numérique et positif
if 'prix' in df.columns:
    print(f'  prix — dtype: {df["prix"].dtype}, min: {df["prix"].min():,.0f}, max: {df["prix"].max():,.0f}')
    print(f'    prix <= 0 : {(df["prix"] <= 0).sum()}')

# Surface : doit être numérique et positive
if 'surface_m2' in df.columns:
    print(f'  surface_m2 — dtype: {df["surface_m2"].dtype}, min: {df["surface_m2"].min()}, max: {df["surface_m2"].max()}')
    print(f'    surface <= 0 : {(df["surface_m2"] <= 0).sum()}')

# Date : format correct ?
if 'date_publication' in df.columns:
    try:
        dates = pd.to_datetime(df['date_publication'])
        print(f'  date_publication — min: {dates.min()}, max: {dates.max()} ✅')
    except:
        print(f'  date_publication — ⚠️ Format de date incohérent')

### 🎯 Observations Étape 1
- **Dimensions** : 1153 annonces × 12 colonnes
- **Variable cible** : `prix` (continue, en MRU → régression)
- **Variables numériques** : surface_m2, nb_chambres, nb_salons, nb_sdb, prix
- **Variables textuelles** : titre, description, caracteristiques (en arabe hassaniya)
- **`nb_sdb` manque dans ~72% des cas** — à traiter en étape 3
- **Source unique** : voursa.com
- **Doublons** : à vérifier

---
## 🧹 Étape 2 : Nettoyage de base

> **Objectif :** Supprimer doublons, corriger/standardiser formats, règles métier, uniformiser catégories.

In [ ]:
# 2.1 — Supprimer les doublons
print(f'Avant : {len(df)} lignes')
df = df.drop_duplicates(subset=[c for c in df.columns if c != 'id'], keep='first').reset_index(drop=True)
print(f'Après : {len(df)} lignes')

In [ ]:
# 2.2 — Standardiser les noms de quartiers (casse, espaces, variantes)
def clean_quartier(name):
    if pd.isna(name):
        return 'Inconnu'
    name = str(name).strip()
    # Mapping des variantes
    mapping = {
        'tevragh zeina': 'Tevragh Zeina', 'tevragh-zeina': 'Tevragh Zeina',
        'ksar': 'Ksar', 'teyarett': 'Teyarett',
        'arafat': 'Arafat', 'dar naim': 'Dar Naim', 'dar-naim': 'Dar Naim',
        'toujounine': 'Toujounine', 'sebkha': 'Sebkha',
        'el mina': 'El Mina', 'elmina': 'El Mina',
        'riyadh': 'Riyadh', 'riyad': 'Riyadh',
    }
    return mapping.get(name.lower(), name)

df['quartier'] = df['quartier'].apply(clean_quartier)

print('Quartiers après nettoyage :')
print(df['quartier'].value_counts())

In [ ]:
# 2.3 — Détecter les incohérences de casse dans les catégorielles
for col in df.select_dtypes(include='object').columns:
    n_raw = df[col].nunique()
    n_lower = df[col].astype(str).str.lower().nunique()
    if n_raw != n_lower:
        print(f'  ⚠️ {col}: {n_raw} catégories brutes vs {n_lower} en minuscules → incohérence de casse')

In [ ]:
# 2.4 — Règles métier : valeurs impossibles
print('🏗️ Vérification des règles métier :')

# Prix : doit être > 0 et raisonnable
if 'prix' in df.columns:
    invalides = (df['prix'] <= 0).sum()
    print(f'  prix <= 0 : {invalides} lignes')
    if invalides > 0:
        df = df[df['prix'] > 0].reset_index(drop=True)
        print(f'    → Supprimées. Nouveau shape : {df.shape}')

# Surface : doit être > 0
if 'surface_m2' in df.columns:
    invalides = (df['surface_m2'] <= 0).sum()
    print(f'  surface_m2 <= 0 : {invalides} lignes')

# nb_chambres : ne peut pas être négatif
for col in ['nb_chambres', 'nb_salons', 'nb_sdb']:
    if col in df.columns:
        neg = (df[col] < 0).sum()
        if neg > 0:
            print(f'  ⚠️ {col} < 0 : {neg} lignes')

print('\n✅ Nettoyage de base terminé')

---
## ❓ Étape 3 : Traitement des Valeurs Manquantes

> **Objectif :** Identifier le mécanisme (MCAR / MAR / MNAR), puis imputer correctement.

| Mécanisme | Description | Exemple dans nos données |
|-----------|-------------|-------------------------|
| **MCAR** | Manquant complètement au hasard | nb_chambres manquant aléatoirement |
| **MAR** | Dépend d'autres variables observées | nb_sdb manquant plus souvent pour les petits biens |
| **MNAR** | Dépend de la valeur manquante elle-même | Prix absent pour les biens très chers ("Prix sur demande") |

In [ ]:
# 3.1 — Vue d'ensemble des valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'N_Missing': missing, 'Pct_Missing': missing_pct})
missing_df = missing_df[missing_df['N_Missing'] > 0].sort_values('Pct_Missing', ascending=False)

print('📊 Valeurs manquantes :')
print(missing_df)
print(f'\nTotal : {missing_df["N_Missing"].sum()} valeurs manquantes sur {df.shape[0] * df.shape[1]} cellules')

In [ ]:
# 3.2 — Visualisation des manquants
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Barplot
if not missing_df.empty:
    missing_df['Pct_Missing'].plot(kind='barh', ax=axes[0], color='coral')
    axes[0].set_xlabel('% Manquant')
    axes[0].set_title('Pourcentage de valeurs manquantes par variable', fontweight='bold')
else:
    axes[0].text(0.5, 0.5, 'Aucune valeur manquante', ha='center', va='center')

# Matrice de nullité (pattern)
cols_with_na = df.columns[df.isnull().any()].tolist()
if cols_with_na:
    axes[1].imshow(df[cols_with_na].isnull().values[:100], aspect='auto', cmap='YlOrRd', interpolation='none')
    axes[1].set_xticks(range(len(cols_with_na)))
    axes[1].set_xticklabels(cols_with_na, rotation=45, ha='right')
    axes[1].set_title('Pattern des manquants (100 premières lignes)', fontweight='bold')
    axes[1].set_ylabel('Index ligne')

plt.tight_layout()
plt.show()

In [ ]:
# 3.3 — Identifier le mécanisme : MCAR, MAR ou MNAR ?

print('=' * 60)
print('ANALYSE DU MÉCANISME DES VALEURS MANQUANTES')
print('=' * 60)

# --- nb_sdb (72% manquant) ---
print('\n📊 nb_sdb (~72% manquant) :')
if 'nb_sdb' in df.columns:
    df['nb_sdb_missing'] = df['nb_sdb'].isna().astype(int)
    
    # Corrélation avec les autres variables numériques
    print('  Corrélation de is_missing avec :')
    for col in ['prix', 'surface_m2', 'nb_chambres', 'nb_salons']:
        if col in df.columns:
            corr = df['nb_sdb_missing'].corr(df[col].fillna(0))
            print(f'    {col:20s} r = {corr:+.4f}')
    
    # Prix moyen quand nb_sdb est manquant vs présent
    prix_missing = df[df['nb_sdb'].isna()]['prix'].median()
    prix_present = df[df['nb_sdb'].notna()]['prix'].median()
    print(f'\n  Prix médian quand nb_sdb manquant : {prix_missing:,.0f} MRU')
    print(f'  Prix médian quand nb_sdb présent  : {prix_present:,.0f} MRU')
    
    # Test statistique
    stat, p = kruskal(
        df[df['nb_sdb'].isna()]['prix'].dropna(),
        df[df['nb_sdb'].notna()]['prix'].dropna()
    )
    print(f'  Kruskal-Wallis H={stat:.2f}, p={p:.4f}')
    if p < 0.05:
        print('  → Différence significative → probablement MAR ou MNAR')
    else:
        print('  → Pas de différence significative → compatible MCAR')

# --- nb_chambres ---
print('\n📊 nb_chambres (~1% manquant) :')
if 'nb_chambres' in df.columns:
    n_miss = df['nb_chambres'].isna().sum()
    print(f'  {n_miss} valeurs manquantes ({100*n_miss/len(df):.1f}%)')
    print(f'  → <5% et probablement MCAR → imputation par médiane acceptable')

# --- caracteristiques ---
print('\n📊 caracteristiques (~14% manquant) :')
if 'caracteristiques' in df.columns:
    n_miss = df['caracteristiques'].isna().sum()
    print(f'  {n_miss} valeurs manquantes ({100*n_miss/len(df):.1f}%)')
    prix_with = df[df['caracteristiques'].notna()]['prix'].median()
    prix_without = df[df['caracteristiques'].isna()]['prix'].median()
    print(f'  Prix médian avec caractéristiques : {prix_with:,.0f} MRU')
    print(f'  Prix médian sans caractéristiques : {prix_without:,.0f} MRU')

In [ ]:
# 3.4 — Appliquer les imputations selon le mécanisme identifié

print('📋 Stratégie d\'imputation :')
print()

# nb_sdb : MAR — indicateur + imputation à 0 (absence de l'info, pas 0 sdb)
print('1. nb_sdb (MAR) :')
print('   → Garder indicateur nb_sdb_missing')
print('   → Imputer à 0 (représente "information non renseignée")')
df['nb_sdb'] = df['nb_sdb'].fillna(0)
print(f'   ✅ Imputé. NaN restants : {df["nb_sdb"].isna().sum()}')

# nb_chambres : MCAR — <5% → imputation par médiane
print('\n2. nb_chambres (MCAR, <5%) :')
med_chambres = df['nb_chambres'].median()
n_imp = df['nb_chambres'].isna().sum()
df['nb_chambres'] = df['nb_chambres'].fillna(med_chambres)
print(f'   → Médiane = {med_chambres}')
print(f'   ✅ {n_imp} valeurs imputées')

# nb_salons : vérifier
if df['nb_salons'].isna().sum() > 0:
    print('\n3. nb_salons :')
    med_salons = df['nb_salons'].median()
    n_imp = df['nb_salons'].isna().sum()
    df['nb_salons'] = df['nb_salons'].fillna(med_salons)
    print(f'   → Médiane = {med_salons}, {n_imp} valeurs imputées ✅')

# caracteristiques : MNAR — ajouter "Inconnu" (absence = pas de caractéristiques renseignées)
print('\n4. caracteristiques (MNAR) :')
print('   → NA signifie "aucune caractéristique renseignée"')
print('   → On garde NA tel quel, on créera un indicateur en feature engineering')

# Nettoyage de la colonne temporaire
df = df.drop(columns=['nb_sdb_missing'], errors='ignore')

print('\n✅ Imputation terminée')
print(f'NaN restants : {df.isnull().sum().sum()} (dans colonnes texte, attendu)')

### 📋 Quand supprimer ?
- **Variable >30-40% NA & faible valeur métier** → On garde `nb_sdb` car l'indicateur de missingness a de la valeur
- **Lignes <5% & MCAR** → `nb_chambres` imputé par médiane
- **Toujours documenter** les choix ✅

---
## 🔍 Étape 4 : Détection d'Outliers

> **Objectif :** IQR, Z-score, visualisation. Distinguer erreurs de mesure vs valeurs extrêmes plausibles.

In [ ]:
# 4.1 — Méthode IQR
def detect_outliers_iqr(series, k=1.5):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - k * IQR, Q3 + k * IQR
    outliers = (series < lower) | (series > upper)
    return outliers, lower, upper

key_cols = ['prix', 'surface_m2', 'nb_chambres', 'nb_salons']
print('📊 Outliers détectés (méthode IQR, k=1.5) :')
print('-' * 60)
for col in key_cols:
    if col in df.columns:
        outliers, lower, upper = detect_outliers_iqr(df[col].dropna())
        n_out = outliers.sum()
        print(f'  {col:20s} → {n_out:>4d} outliers ({100*n_out/len(df):.1f}%) | Bornes: [{lower:,.0f}, {upper:,.0f}]')

In [ ]:
# 4.2 — Z-score
print('📊 Outliers détectés (Z-score, |z| > 3) :')
print('-' * 60)
for col in key_cols:
    if col in df.columns:
        z = np.abs(stats.zscore(df[col].dropna()))
        n_out = (z > 3).sum()
        print(f'  {col:20s} → {n_out:>4d} outliers ({100*n_out/len(df[col].dropna()):.1f}%)')

In [ ]:
# 4.3 — Boxplots
fig, axes = plt.subplots(1, len(key_cols), figsize=(5*len(key_cols), 5))
for i, col in enumerate(key_cols):
    if col in df.columns:
        sns.boxplot(y=df[col], ax=axes[i], color='lightblue',
                    flierprops=dict(markerfacecolor='coral', markersize=4))
        axes[i].set_title(col, fontweight='bold')
plt.suptitle('Boxplots — Détection des outliers', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 4.4 — Scatter prix vs surface — identifier les outliers suspects
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['surface_m2'], df['prix']/1e6, alpha=0.4, s=15, color='steelblue')

# Marquer les cas extrêmes
extreme = df[(df['prix'] > 30e6) | ((df['surface_m2'] < 250) & (df['prix'] > 40e6))]
if len(extreme) > 0:
    ax.scatter(extreme['surface_m2'], extreme['prix']/1e6, color='red', s=50, zorder=5, label='Suspects')
    for _, row in extreme.iterrows():
        ax.annotate(f'id={row["id"]}', (row['surface_m2'], row['prix']/1e6), fontsize=8, color='red')

ax.set_xlabel('Surface (m²)')
ax.set_ylabel('Prix (millions MRU)')
ax.set_title('Prix vs Surface — Outliers', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print('\n🔍 Cas suspects (prix > 30M) :')
print(df[df['prix'] > 30e6][['id', 'prix', 'surface_m2', 'quartier', 'nb_chambres']].to_string())

In [ ]:
# 4.5 — Catégoriel : classes rares
print('📊 Classes rares (<10 observations) :')
for col in df.select_dtypes(include='object').columns:
    if col not in ['titre', 'description', 'caracteristiques']:
        counts = df[col].value_counts()
        rare = counts[counts < 10]
        if len(rare) > 0:
            print(f'  {col}: {rare.to_dict()}')

In [ ]:
# 4.6 — Décision sur les outliers
print('📋 Décisions sur les outliers :')
print()

# Cas 1: prix extrêmes mais dans Tevragh Zeina → plausible (quartier de luxe)
extreme_tz = df[(df['prix'] > 30e6) & (df['quartier'] == 'Tevragh Zeina')]
print(f'  Prix > 30M dans Tevragh Zeina : {len(extreme_tz)} → GARDER (quartier huppé, plausible)')

# Cas 2 : petite surface avec prix très élevé → suspect
suspect = df[(df['surface_m2'] < 200) & (df['prix'] > 40e6)]
print(f'  Surface < 200m² et prix > 40M : {len(suspect)} → SUSPECT (erreur possible)')

# Cas 3 : nb_salons > 10 → probablement une erreur de saisie
print(f'  nb_salons > 10 : {(df["nb_salons"] > 10).sum()} → Capper à 10')
df['nb_salons'] = df['nb_salons'].clip(upper=10)
df['nb_chambres'] = df['nb_chambres'].clip(upper=15)

print()
print('→ Stratégie : on garde les outliers (modèles à base d\'arbres sont robustes)')
print('→ On applique un clipping léger sur nb_salons et nb_chambres')
print('→ Le prix sera prédit en log-space (atténue l\'effet des extrêmes)')

---
## 📊 Étape 5 : Analyse Univariée

> **Objectif :** Distribution de chaque variable — histogrammes, KDE, stats descriptives, test de normalité.

In [ ]:
# 5.1 — Variable cible : prix
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogramme brut
sns.histplot(df['prix'], bins=50, kde=True, ax=axes[0], color='steelblue')
axes[0].axvline(df['prix'].mean(), color='red', linestyle='--', label=f'Moyenne: {df["prix"].mean():,.0f}')
axes[0].axvline(df['prix'].median(), color='green', linestyle='--', label=f'Médiane: {df["prix"].median():,.0f}')
axes[0].set_title('Distribution du Prix (brut)', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
axes[0].legend(fontsize=9)

# Log-transformé
sns.histplot(np.log1p(df['prix']), bins=50, kde=True, ax=axes[1], color='coral')
axes[1].set_title('Distribution du log(1+Prix)', fontweight='bold')

# QQ-plot
stats.probplot(np.log1p(df['prix']), dist="norm", plot=axes[2])
axes[2].set_title('QQ-Plot (log-prix vs Normal)', fontweight='bold')

plt.suptitle('Variable cible : Prix', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Stats
print(f'Moyenne  : {df["prix"].mean():>12,.0f} MRU')
print(f'Médiane  : {df["prix"].median():>12,.0f} MRU')
print(f'Std      : {df["prix"].std():>12,.0f} MRU')
print(f'Skewness : {df["prix"].skew():>12.2f} (asymétrie à droite)')
print(f'Kurtosis : {df["prix"].kurtosis():>12.2f}')

# Shapiro-Wilk sur log(prix)
stat, p = shapiro(np.log1p(df['prix'].sample(min(500, len(df)), random_state=42)))
print(f'\nShapiro-Wilk (log-prix) : W={stat:.4f}, p={p:.4f}')
print(f'  → {"Distribution approximativement normale ✅" if p > 0.05 else "Pas parfaitement normale, mais log-transform améliore"}')

In [ ]:
# 5.2 — Variables numériques continues
num_plot = ['surface_m2', 'nb_chambres', 'nb_salons', 'nb_sdb']
num_plot = [c for c in num_plot if c in df.columns]

fig, axes = plt.subplots(2, len(num_plot), figsize=(5*len(num_plot), 10))

for i, col in enumerate(num_plot):
    # Histogramme
    sns.histplot(df[col].dropna(), bins=30, kde=True, ax=axes[0, i], color='steelblue')
    axes[0, i].set_title(f'{col}', fontweight='bold')
    axes[0, i].axvline(df[col].median(), color='green', linestyle='--', alpha=0.7)
    
    # Boxplot
    sns.boxplot(y=df[col].dropna(), ax=axes[1, i], color='lightblue')
    axes[1, i].set_title(f'{col} (boxplot)')

plt.suptitle('Analyse Univariée — Variables Numériques', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Stats
print('📊 Statistiques descriptives :')
print(df[num_plot].describe().round(2))

In [ ]:
# 5.3 — Variable catégorielle : quartier
fig, ax = plt.subplots(figsize=(10, 5))
df['quartier'].value_counts().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Distribution des annonces par quartier', fontweight='bold')
ax.set_xlabel('Nombre d\'annonces')
plt.tight_layout()
plt.show()

print('Répartition :')
for q, n in df['quartier'].value_counts().items():
    print(f'  {q:20s} {n:>5d} ({100*n/len(df):.1f}%)')

---
## 📈 Étape 6 : Analyse Bivariée

> **Objectif :** Relations entre 2 variables — Num×Num (scatter, corrélation), Num×Cat (boxplot, ANOVA/Kruskal), Cat×Cat (Chi²).

In [ ]:
# 6.1 — Num × Num : Scatter plots vs Prix
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
scatter_vars = ['surface_m2', 'nb_chambres', 'nb_salons', 'nb_sdb', 'nb_chambres', 'surface_m2']
y_var = 'prix'

for i, col in enumerate(['surface_m2', 'nb_chambres', 'nb_salons']):
    ax = axes[0, i]
    ax.scatter(df[col], df['prix']/1e6, alpha=0.3, s=15, color='steelblue')
    ax.set_xlabel(col)
    ax.set_ylabel('Prix (M MRU)')
    
    # Corrélation
    r = df[col].corr(df['prix'])
    ax.set_title(f'{col} vs Prix (r={r:.3f})', fontweight='bold')

# Log-space scatter
for i, col in enumerate(['surface_m2', 'nb_chambres', 'nb_salons']):
    ax = axes[1, i]
    ax.scatter(np.log1p(df[col]), np.log1p(df['prix']), alpha=0.3, s=15, color='coral')
    ax.set_xlabel(f'log(1+{col})')
    ax.set_ylabel('log(1+prix)')
    
    r = np.log1p(df[col]).corr(np.log1p(df['prix']))
    ax.set_title(f'Log-space (r={r:.3f})', fontweight='bold')

plt.suptitle('Analyse Bivariée — Num × Num', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 6.2 — Num × Cat : Boxplots prix par quartier
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Prix par quartier
order = df.groupby('quartier')['prix'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='quartier', y='prix', order=order, ax=axes[0], palette='coolwarm')
axes[0].set_title('Prix par Quartier', fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

# Violin plot
sns.violinplot(data=df, x='quartier', y='prix', order=order, ax=axes[1], palette='coolwarm', inner='quartile')
axes[1].set_title('Prix par Quartier (violin)', fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

plt.tight_layout()
plt.show()

In [ ]:
# 6.3 — Test statistique : Kruskal-Wallis (non-paramétrique)
print('🧪 Tests statistiques — Prix par Quartier :')
print('-' * 50)

groups = [g['prix'].dropna().values for _, g in df.groupby('quartier')]
stat, p = kruskal(*groups)
print(f'Kruskal-Wallis H = {stat:.2f}, p = {p:.2e}')
print(f'→ {"Différence significative entre quartiers ✅" if p < 0.05 else "Pas de différence significative"}')

# Taille d'effet
print('\nPrix médian par quartier :')
for q in order:
    med = df[df['quartier'] == q]['prix'].median()
    n = len(df[df['quartier'] == q])
    print(f'  {q:20s} {med:>12,.0f} MRU (n={n})')

In [ ]:
# 6.4 — Cat × Cat : Test Chi² (quartier × has_garage extrait)
# Extraire has_garage depuis les caractéristiques
df['has_garage_temp'] = df['caracteristiques'].str.contains('Garage', case=False, na=False).astype(str)

ct = pd.crosstab(df['quartier'], df['has_garage_temp'])
chi2, p, dof, expected = chi2_contingency(ct)

print('📊 Chi² : Quartier × Garage')
print(f'  Chi² = {chi2:.2f}, p = {p:.4f}, dof = {dof}')
print(f'  → {"Association significative ✅" if p < 0.05 else "Pas d\'association significative"}')
print('\nTable de contingence :')
print(ct)

df = df.drop(columns=['has_garage_temp'])

---
## 🧮 Étape 7 : Analyse Multivariée

> **Objectif :** Heatmap de corrélations, VIF (multicollinéarité), PCA, Pairplots.

In [ ]:
# 7.1 — Matrice de corrélation
num_df = df.select_dtypes(include='number').drop(columns=['id'], errors='ignore')
corr_matrix = num_df.corr()

# Top corrélations avec prix
print('📊 Top corrélations avec le prix :')
prix_corr = corr_matrix['prix'].abs().sort_values(ascending=False)
for col, val in prix_corr.head(10).items():
    if col != 'prix':
        print(f'  {col:25s} {corr_matrix.loc[col, "prix"]:+.4f}')

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title('Matrice de Corrélation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 7.2 — VIF (Variance Inflation Factor)
from sklearn.preprocessing import StandardScaler

vif_cols = ['surface_m2', 'nb_chambres', 'nb_salons', 'nb_sdb']
vif_cols = [c for c in vif_cols if c in df.columns]

X_vif = df[vif_cols].dropna()
X_scaled = StandardScaler().fit_transform(X_vif)

# Calcul VIF
from numpy.linalg import inv
corr_vif = np.corrcoef(X_scaled.T)
try:
    vif_values = np.diag(inv(corr_vif))
    print('📊 VIF (Variance Inflation Factor) :')
    print('  (VIF > 5 = multicollinéarité problématique)')
    for col, vif in zip(vif_cols, vif_values):
        flag = ' ⚠️' if vif > 5 else ' ✅'
        print(f'  {col:20s} VIF = {vif:.2f}{flag}')
except:
    print('⚠️ Matrice singulière — VIF non calculable')

In [ ]:
# 7.3 — PCA (visualisation en 2D)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pca_features = ['surface_m2', 'nb_chambres', 'nb_salons', 'nb_sdb']
pca_features = [c for c in pca_features if c in df.columns]

X_pca = df[pca_features].dropna()
X_scaled = StandardScaler().fit_transform(X_pca)

pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(pca_result[:, 0], pca_result[:, 1],
                     c=np.log1p(df.loc[X_pca.index, 'prix']), cmap='coolwarm',
                     alpha=0.5, s=15)
plt.colorbar(scatter, label='log(1+prix)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('PCA — Projection 2D colorée par prix', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Variance expliquée : PC1={pca.explained_variance_ratio_[0]*100:.1f}%, PC2={pca.explained_variance_ratio_[1]*100:.1f}%')

In [ ]:
# 7.4 — Pairplot (échantillon)
sample = df[['prix', 'surface_m2', 'nb_chambres', 'nb_salons', 'quartier']].sample(
    min(300, len(df)), random_state=42
)

# Simplifier quartier pour la lisibilité
top_quartiers = ['Tevragh Zeina', 'Teyarett', 'Arafat', 'Toujounine']
sample['quartier_simple'] = sample['quartier'].where(sample['quartier'].isin(top_quartiers), 'Autre')

g = sns.pairplot(sample, hue='quartier_simple', diag_kind='kde', 
                 plot_kws={'alpha': 0.5, 's': 20},
                 palette={'Tevragh Zeina': 'red', 'Teyarett': 'blue', 
                          'Arafat': 'green', 'Toujounine': 'orange', 'Autre': 'gray'})
g.fig.suptitle('Pairplot — Variables principales par quartier', y=1.02, fontweight='bold')
plt.show()

---
## ⚙️ Étape 8 : Préparation Finale

> **Objectif :** Encodage, normalisation, split train/test, pipeline reproductible. Prêt pour la Phase 4 (Modélisation).

In [ ]:
# 8.1 — Feature Engineering de base
print('📋 Features créées pour la modélisation :')

# Log-transform du prix (cible)
df['log_prix'] = np.log1p(df['prix'])
print('  log_prix = log(1 + prix)')

# Nombre total de pièces
df['nb_pieces_total'] = df['nb_chambres'] + df['nb_salons']
print('  nb_pieces_total = nb_chambres + nb_salons')

# Surface par pièce
df['surface_par_piece'] = df['surface_m2'] / (df['nb_pieces_total'] + 1)
print('  surface_par_piece = surface / (nb_pieces + 1)')

# Log surface
df['log_surface'] = np.log1p(df['surface_m2'])
print('  log_surface = log(1 + surface_m2)')

# Indicateurs depuis les caractéristiques
df['has_titre_foncier'] = df['caracteristiques'].str.contains('Titre foncier', case=False, na=False).astype(int)
df['has_garage'] = df['caracteristiques'].str.contains('Garage', case=False, na=False).astype(int)
print('  has_titre_foncier, has_garage (depuis caractéristiques)')

# Âge de l'annonce
df['date_publication'] = pd.to_datetime(df['date_publication'])
df['age_annonce_jours'] = (df['date_publication'].max() - df['date_publication']).dt.days
print('  age_annonce_jours')

print(f'\nShape final : {df.shape}')

In [ ]:
# 8.2 — Encodage des variables catégorielles
from sklearn.preprocessing import LabelEncoder

# Quartier : Target Encoding serait idéal (Phase 4), pour l'instant Label Encoding
le = LabelEncoder()
df['quartier_encoded'] = le.fit_transform(df['quartier'])
print('Encodage quartier :')
for i, q in enumerate(le.classes_):
    print(f'  {q} → {i}')

In [ ]:
# 8.3 — Normalisation / Standardisation (aperçu)
from sklearn.preprocessing import StandardScaler

features_to_scale = ['surface_m2', 'nb_chambres', 'nb_salons', 'nb_sdb']

print('📊 Avant standardisation :')
print(df[features_to_scale].describe().loc[['mean', 'std']].round(2))

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[features_to_scale] = scaler.fit_transform(df[features_to_scale])

print('\n📊 Après standardisation :')
print(df_scaled[features_to_scale].describe().loc[['mean', 'std']].round(2))
print('\n→ La standardisation sera appliquée dans le pipeline de modélisation (Phase 4)')

In [ ]:
# 8.4 — Split Train/Test (pour validation)
from sklearn.model_selection import train_test_split

feature_cols = ['surface_m2', 'nb_chambres', 'nb_salons', 'nb_sdb',
                'quartier_encoded', 'has_titre_foncier', 'has_garage',
                'nb_pieces_total', 'surface_par_piece', 'log_surface', 'age_annonce_jours']

X = df[feature_cols]
y = df['log_prix']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Split train/test (80/20) :')
print(f'  X_train : {X_train.shape}')
print(f'  X_test  : {X_test.shape}')
print(f'\n⚠️ Note : pour la compétition Kaggle, on utilise le jeu de test fourni.')
print(f'  Ce split est pour la validation locale uniquement.')

In [ ]:
# 8.5 — Sauvegarder le dataset préparé
output_path = 'prepared_data.csv'
df.to_csv(output_path, index=False)
print(f'✅ Dataset préparé sauvegardé : {output_path}')
print(f'   Shape : {df.shape}')
print(f'   Colonnes : {df.columns.tolist()}')

---
## 📝 Documentation — Livrable

### Décisions prises

| Étape | Décision | Justification |
|-------|----------|---------------|
| Doublons | Supprimés (keep='first') | Annonces identiques |
| Quartiers | Standardisés (casse, variantes) | 'tevragh zeina' = 'Tevragh Zeina' |
| `nb_sdb` (72% NA) | Imputé à 0 + indicateur missing | MAR : lié au type de bien. L'indicateur est informatif |
| `nb_chambres` (<2% NA) | Médiane | MCAR, trop peu pour supprimer |
| `caracteristiques` (14% NA) | Gardé NA | MNAR : absence = pas de caractéristiques |
| Outliers prix | Gardés | Plausibles (Tevragh Zeina = quartier luxe). Modèle en log-space |
| `nb_salons` > 10 | Clippé à 10 | Erreur de saisie probable |
| Prix | log1p transform | Asymétrie forte (skew > 1), QQ-plot amélioré |

### Hypothèses & Limites
- Les coordonnées sont au niveau quartier (pas adresse exacte)
- Les descriptions en arabe hassaniya contiennent de l'information non exploitée dans cette phase
- Le dataset est petit (1153 annonces) → risque d'overfit avec trop de features

### Dictionnaire de données (post-EDA)

| Variable | Type | Description |
|----------|------|-------------|
| `prix` | float | Prix de vente en MRU (cible) |
| `log_prix` | float | log(1+prix), cible transformée |
| `surface_m2` | float | Surface en m² |
| `nb_chambres` | float | Nombre de chambres (imputé) |
| `nb_salons` | float | Nombre de salons |
| `nb_sdb` | float | Nombre de salles de bain (imputé à 0) |
| `quartier` | str | Quartier standardisé |
| `quartier_encoded` | int | Quartier encodé (Label) |
| `nb_pieces_total` | float | nb_chambres + nb_salons |
| `surface_par_piece` | float | surface / (nb_pieces + 1) |
| `log_surface` | float | log(1+surface) |
| `has_titre_foncier` | int | Titre foncier mentionné (0/1) |
| `has_garage` | int | Garage mentionné (0/1) |
| `age_annonce_jours` | int | Jours depuis la publication |

---

### Prochaine étape : Phase 4 — Feature Engineering & Modélisation (`04_modeling.ipynb`)

---
*🎓 Projet Capstone — Master 1 Machine Learning — SupNum 2026*